### Python's copy Module: Shallow vs Deep Copy
**The Core Concept First**

In Python, assignment (`=`) **doesn't** copy objects — it creates a **new reference** to the same object. The copy module solves this.

In [1]:
# Assignment = same object, NOT a copy
a = [1, 2, [3, 4]]
b = a
b[0] = 99
print(a)  # [99, 2, [3, 4]] ← a was mutated!

[99, 2, [3, 4]]


#### 1. Shallow Copy — `copy.copy()`
Copies the **outer object only**. Nested objects are still **shared references**.

In [2]:
import copy

original = [1, 2, [3, 4]]
shallow = copy.copy(original)

In [3]:
id(original)

2164340508096

In [4]:
id(shallow)

2164340276096

In [6]:
# Modifying a top-level element → safe
shallow[0] = 99
shallow

[99, 2, [3, 4]]

In [7]:
original  # [1, 2, [3, 4]]  ✅ not affected

[1, 2, [3, 4]]

In [ ]:
# Modifying a nested element → DANGEROUS
shallow[2][0] = 999
shallow  # [1, 2, [999, 4]]

[99, 2, [999, 4]]

In [12]:
original  # ❌ original mutated!

[1, 2, [999, 4]]

Memory Model — Shallow Copy:

```shell
original ──→ [ 1 | 2 | ──→ [3, 4] ]
                              ↑
shallow  ──→ [ 1 | 2 | ───────┘    ]  ← shared nested ref!
```

---

#### 2. Deep Copy — `copy.deepcopy()`
**Recursively copies everything** — no shared references at any level.

In [13]:
import copy

original = [1, 2, [3, 4]]
deep = copy.deepcopy(original)

In [16]:
# Modifying nested element → fully isolated
deep[2][0] = 999
deep

[1, 2, [999, 4]]

In [17]:
original  # [1, 2, [3, 4]]  ✅ completely safe

[1, 2, [3, 4]]

Memory Model — Deep Copy:
```shell
original ──→ [ 1 | 2 | ──→ [3, 4]    ]
deep     ──→ [ 1 | 2 | ──→ [3, 4]    ]  ← brand new nested list!
```

---

#### 3. Real-World Object Example (OOP Context)

In [18]:
import copy

class Address:
    def __init__(self, city):
        self.city = city

class User:
    def __init__(self, name, address):
        self.name = name
        self.address = address   # nested mutable object

In [19]:
user1 = User("Alice", Address("Prague"))

In [20]:
# --- Shallow copy ---
user2 = copy.copy(user1)
user2.name = "Bob"           # safe — primitive attribute
user2.address.city = "Brno"  # ❌ mutates user1.address too!

In [21]:
print(user1.name)          # "Alice" ✅

Alice


In [22]:
print(user1.address.city)  # "Brno"  ❌ — shared Address object

Brno


In [23]:
# --- Deep copy ---
user3 = copy.deepcopy(user1)
user3.address.city = "Ostrava"

In [24]:
print(user1.address.city)  # "Brno"  ✅ — fully isolated

Brno


#### 4. `__copy__` and `__deepcopy__` — Custom Behavior
You can control copy behavior in your own classes:

In [25]:
import copy

class Config:
    def __init__(self, settings, secret_key):
        self.settings = settings
        self.secret_key = secret_key  # should NOT be deep-copied

    def __copy__(self):
        # Custom shallow: share secret_key reference intentionally
        new = Config(self.settings.copy(), self.secret_key)
        return new

    def __deepcopy__(self, memo):
        # Custom deep: copy settings fully, but keep secret_key shared
        new = Config(copy.deepcopy(self.settings, memo), self.secret_key)
        memo[id(self)] = new
        return new

`memo` is a dictionary that **tracks already-copied objects** to handle circular references and avoid infinite recursion.

---

#### 5. Circular Reference Handling
`deepcopy` handles it — shallow copy **doesn't need** to since it **doesn't recurse**.

In [26]:
import copy

a = [1, 2]
a.append(a)  # circular reference! a → [..., a]

In [27]:
shallow = copy.copy(a)      # ✅ works (doesn't recurse)

In [29]:
deep = copy.deepcopy(a)  # ✅ works (memo dict prevents infinite loop)

# Without memo, deepcopy would hit infinite recursion

#### 6. Performance & When to Use What

| **Scenario**                             | **Use**                             |
| ---------------------------------------- | ------------------------------- |
| Flat `list`/`dict`, no nested mutables   | `copy.copy()`                   |
| Nested mutable structures                | `copy.deepcopy()`               |
| Large objects (performance critical)     | `copy.copy()` + manual handling |
| Config objects, game state, undo systems | `copy.deepcopy()`               |
| Django model instances (careful!)        | Neither — use `pk=None` trick   |

---

### 🎯 Tricky Interview Questions

#### ❓ Q1: Does `list.copy()` behave like **shallow** or **deep** copy?

In [30]:
a = [[1, 2], [3, 4]]

In [31]:
b = a.copy()       # also: b = a[:]  or  b = list(a)
b[0][0] = 99
print(a)  # [[99, 2], [3, 4]] ← SHALLOW, nested objects shared

[[99, 2], [3, 4]]


Answer: It's a **shallow copy** — same as `copy.copy()`.

---

#### ❓ Q2: What happens with immutable nested objects?

In [32]:
import copy

original = (1, 2, [3, 4])
deep = copy.deepcopy(original)

In [33]:
print(original is deep)  # True or False?

False


**Answer**: `False` — deepcopy **always** creates **new objects**, even for tuples. BUT for a tuple of only immutables like (1, 2, 3), Python may optimize and return the same object **since immutables can't change**.

---

#### ❓ Q3: Is this a copy?

In [ ]:
import copy

d = {"a": 1, "b": [2, 3]}
d2 = {**d}  # dict unpacking is shallow

In [38]:
id(d)

2164340511104

In [39]:
id(d2)

2164340633792

In [37]:
d2

{'a': 1, 'b': [2, 3, 99]}

In [35]:
d2["b"].append(99)

In [36]:
print(d["b"])  # ?

[2, 3, 99]


**Answer**: `[2, 3, 99]` — **dict unpacking is shallow**, nested list is shared.

---

#### ❓ Q4: deepcopy of a Django model instance — what's the pitfall?

In [ ]:
import copy
obj = MyModel.objects.get(pk=1)
obj_copy = copy.deepcopy(obj)
obj_copy.save()  # What happens? - Updates the existing record.

**Answer**: It **updates the existing record** (`pk=1` still exists). To create a new DB record, you must set `obj_copy.pk = None` before saving.

---

#### ❓ Q5: Why can deepcopy fail?

In [40]:
import copy, threading

lock = threading.Lock()
data = {"lock": lock, "value": 42}
copy.deepcopy(data)  # Raises TypeError!

TypeError: cannot pickle '_thread.lock' object

**Answer**: Some objects like **locks**, **file handles**, **sockets**, **database connections** are not copyable. You handle this via `__deepcopy__` to exclude them.

---

### 🧠 The One-Line Mental Model

> **Shallow copy** = **new box**, **same items** inside.

> **Deep copy** = **new box**, completely **new copies** of everything inside, recursively.